In [ ]:
!pip -q install pandas requests tqdm pillow scikit-learn

In [ ]:
import os
import re
import time
import json
import hashlib
import random
import shutil
from pathlib import Path

import pandas as pd
import requests
from tqdm.auto import tqdm
from PIL import Image, ImageFile
from sklearn.model_selection import train_test_split

ImageFile.LOAD_TRUNCATED_IMAGES = True

# -----------------------------
# Main controls
# -----------------------------
PROJECT_DIR = Path("/content/lichen_growthform_silver")
RAW_IMG_DIR = PROJECT_DIR / "images_raw"
SPLIT_IMG_DIR = PROJECT_DIR / "images_split"
META_PATH = PROJECT_DIR / "metadata.csv"

# Start small first. Increase later.
TARGET_PER_CLASS = 700          # total downloaded images per growth form
MAX_IMAGES_PER_SPECIES = 8      # prevents one species from dominating the dataset
REQUEST_SLEEP = 0.25            # be polite to servers

# Keep this True for student/research use.
# For a future commercial app, set this False and only use CC0/CC-BY images.
ALLOW_CC_BY_NC = True

# iNaturalist Research-grade Observations dataset on GBIF
INAT_RG_DATASET_KEY = "50c9509d-22c7-4a22-a47d-8c48425ef4a7"

# Zenodo current CSV from the lichen attribute dataset
TRAIT_CSV_URL = "https://zenodo.org/records/14236370/files/Lichen_dataset2024.11.28.csv?download=1"

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
RAW_IMG_DIR.mkdir(parents=True, exist_ok=True)

for label in ["crustose", "foliose", "fruticose"]:
    (RAW_IMG_DIR / label).mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)

In [ ]:
trait_csv_path = PROJECT_DIR / "Lichen_dataset2024.11.28.csv"

if not trait_csv_path.exists():
    print("Downloading lichen trait dataset...")
    r = requests.get(TRAIT_CSV_URL, timeout=120)
    r.raise_for_status()
    trait_csv_path.write_bytes(r.content)

traits = pd.read_csv(trait_csv_path)

print("Columns:")
print(traits.columns.tolist())
print("\nShape:", traits.shape)
traits.head()

In [ ]:
def find_col(df, possible_names):
    lower_map = {c.lower().strip(): c for c in df.columns}
    for name in possible_names:
        key = name.lower().strip()
        if key in lower_map:
            return lower_map[key]
    raise ValueError(f"Could not find any of these columns: {possible_names}")

species_col = find_col(traits, ["Scientific name", "scientificName", "species", "Species"])
growth_col = find_col(traits, ["Growth form", "growth_form", "growthForm", "Growth Form"])

clean = traits[[species_col, growth_col]].copy()
clean.columns = ["scientific_name", "growth_form"]

clean["scientific_name"] = clean["scientific_name"].astype(str).str.strip()
clean["growth_form"] = clean["growth_form"].astype(str).str.strip().str.lower()

# Keep the 3 main CV classes only
valid_forms = {"crustose", "foliose", "fruticose"}
clean = clean[clean["growth_form"].isin(valid_forms)]
clean = clean[clean["scientific_name"].notna()]
clean = clean[clean["scientific_name"].str.len() > 3]

# If a species appears many times, keep the majority growth-form label
species_label = (
    clean.groupby(["scientific_name", "growth_form"])
    .size()
    .reset_index(name="count")
    .sort_values(["scientific_name", "count"], ascending=[True, False])
    .drop_duplicates("scientific_name")
    .reset_index(drop=True)
)

print("Unique species with usable growth-form labels:", len(species_label))
print(species_label["growth_form"].value_counts())
species_label.head()

In [ ]:
GBIF_SEARCH_URL = "https://api.gbif.org/v1/occurrence/search"

session = requests.Session()
session.headers.update({
    "User-Agent": "EcoSignal-growth-form-dataset-builder/1.0 student research contact: example@example.com"
})

def normalize_license(license_text):
    if not license_text:
        return ""
    return str(license_text).lower()

def license_allowed(license_text):
    """
    Allows CC0, CC-BY, and optionally CC-BY-NC.
    Stores exact license in metadata for attribution.
    """
    lic = normalize_license(license_text)

    cc0 = "cc0" in lic or "publicdomain/zero" in lic
    cc_by = "by/4.0" in lic or "licenses/by/" in lic or "cc_by" in lic
    cc_by_nc = "by-nc" in lic or "cc_by_nc" in lic

    if cc0:
        return True
    if cc_by and not cc_by_nc:
        return True
    if ALLOW_CC_BY_NC and cc_by_nc:
        return True

    return False

def safe_filename(text):
    text = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text))
    return text[:120]

def gbif_search_species(scientific_name, limit=300, offset=0):
    params = {
        "datasetKey": INAT_RG_DATASET_KEY,
        "scientificName": scientific_name,
        "mediaType": "StillImage",
        "basisOfRecord": "HUMAN_OBSERVATION",
        "limit": limit,
        "offset": offset
    }

    r = session.get(GBIF_SEARCH_URL, params=params, timeout=60)
    r.raise_for_status()
    return r.json()

def extract_still_images(record):
    """
    Returns image media items from a GBIF occurrence record.
    """
    media = record.get("media", []) or []
    stills = []

    for m in media:
        mtype = str(m.get("type", "")).lower()
        identifier = m.get("identifier")

        if not identifier:
            continue

        # Some records say StillImage, some simply include image URLs.
        looks_like_image = any(
            str(identifier).lower().split("?")[0].endswith(ext)
            for ext in [".jpg", ".jpeg", ".png", ".webp"]
        )

        if "stillimage" in mtype or "image" in mtype or looks_like_image:
            stills.append(m)

    return stills

def make_gbif_cache_url(gbif_key, image_url, width=900):
    """
    GBIF image cache URL. This downloads a resized image when available.
    Falls back to original URL if cache fails.
    """
    md5 = hashlib.md5(image_url.encode("utf-8")).hexdigest()
    return f"https://api.gbif.org/v1/image/cache/{width}x/occurrence/{gbif_key}/media/{md5}"

def download_and_validate_image(url, out_path):
    """
    Downloads image, validates with PIL, converts to RGB JPEG.
    """
    try:
        r = session.get(url, timeout=45, stream=True)
        if r.status_code != 200:
            return False, f"HTTP {r.status_code}"

        temp_path = str(out_path) + ".tmp"
        with open(temp_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

        img = Image.open(temp_path)
        img = img.convert("RGB")

        # Remove tiny images
        if img.width < 180 or img.height < 180:
            os.remove(temp_path)
            return False, f"too small: {img.width}x{img.height}"

        # Resize very large images to keep Colab storage manageable
        max_side = 1200
        if max(img.size) > max_side:
            img.thumbnail((max_side, max_side))

        img.save(out_path, "JPEG", quality=92)
        os.remove(temp_path)
        return True, "ok"

    except Exception as e:
        try:
            if os.path.exists(str(out_path) + ".tmp"):
                os.remove(str(out_path) + ".tmp")
        except:
            pass
        return False, str(e)

In [ ]:
random.seed(42)

metadata_rows = []
seen_image_urls = set()
seen_gbif_keys = set()

# Prioritize species with more records in the trait table
species_priority = (
    clean.groupby(["scientific_name", "growth_form"])
    .size()
    .reset_index(name="trait_record_count")
    .sort_values("trait_record_count", ascending=False)
)

# Shuffle within each class so we do not always hit the same genera first
species_by_class = {}
for label in ["crustose", "foliose", "fruticose"]:
    sub = species_priority[species_priority["growth_form"] == label].copy()
    sub = sub.sample(frac=1.0, random_state=42).reset_index(drop=True)
    species_by_class[label] = sub["scientific_name"].tolist()

download_counts = {label: 0 for label in ["crustose", "foliose", "fruticose"]}

for label in ["crustose", "foliose", "fruticose"]:
    print(f"\n=== Downloading {label} images ===")

    for scientific_name in tqdm(species_by_class[label]):
        if download_counts[label] >= TARGET_PER_CLASS:
            break

        species_downloaded = 0

        try:
            data = gbif_search_species(scientific_name, limit=300, offset=0)
            records = data.get("results", [])
        except Exception as e:
            print("GBIF search failed:", scientific_name, e)
            continue

        random.shuffle(records)

        for rec in records:
            if download_counts[label] >= TARGET_PER_CLASS:
                break
            if species_downloaded >= MAX_IMAGES_PER_SPECIES:
                break

            gbif_key = rec.get("key")
            if gbif_key in seen_gbif_keys:
                continue

            media_items = extract_still_images(rec)
            if not media_items:
                continue

            for m in media_items:
                image_url = m.get("identifier")
                if not image_url or image_url in seen_image_urls:
                    continue

                media_license = m.get("license") or rec.get("license") or ""
                if not license_allowed(media_license):
                    continue

                # Build output path
                image_hash = hashlib.md5(image_url.encode("utf-8")).hexdigest()[:12]
                fname = f"{label}_{safe_filename(scientific_name)}_{gbif_key}_{image_hash}.jpg"
                out_path = RAW_IMG_DIR / label / fname

                # Try GBIF resized cache first, then original media URL
                cache_url = make_gbif_cache_url(gbif_key, image_url, width=900)

                ok, msg = download_and_validate_image(cache_url, out_path)
                if not ok:
                    ok, msg = download_and_validate_image(image_url, out_path)

                time.sleep(REQUEST_SLEEP)

                if not ok:
                    continue

                seen_image_urls.add(image_url)
                seen_gbif_keys.add(gbif_key)
                species_downloaded += 1
                download_counts[label] += 1

                metadata_rows.append({
                    "file_path": str(out_path),
                    "file_name": fname,
                    "growth_form_label": label,
                    "label_type": "silver_species_trait_join",
                    "scientific_name_trait": scientific_name,
                    "gbif_key": gbif_key,
                    "gbif_scientific_name": rec.get("scientificName"),
                    "gbif_accepted_scientific_name": rec.get("acceptedScientificName"),
                    "gbif_species": rec.get("species"),
                    "country": rec.get("country"),
                    "stateProvince": rec.get("stateProvince"),
                    "decimalLatitude": rec.get("decimalLatitude"),
                    "decimalLongitude": rec.get("decimalLongitude"),
                    "eventDate": rec.get("eventDate"),
                    "basisOfRecord": rec.get("basisOfRecord"),
                    "datasetKey": rec.get("datasetKey"),
                    "publisher": rec.get("publisher"),
                    "image_url": image_url,
                    "media_license": media_license,
                    "media_creator": m.get("creator"),
                    "media_rightsHolder": m.get("rightsHolder"),
                    "media_title": m.get("title"),
                    "source": "GBIF iNaturalist Research-grade + Zenodo lichen trait table"
                })

                break

    print("Current counts:", download_counts)

metadata = pd.DataFrame(metadata_rows)
metadata.to_csv(META_PATH, index=False)

print("\nFinished.")
print(download_counts)
print("Metadata saved to:", META_PATH)
metadata.head()

In [ ]:
metadata = pd.read_csv(META_PATH)
print(metadata["growth_form_label"].value_counts())

metadata[[
    "file_name",
    "growth_form_label",
    "scientific_name_trait",
    "gbif_scientific_name",
    "media_license",
    "media_creator",
    "image_url"
]].head()

In [ ]:
import os
import shutil
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
import ipywidgets as widgets

PROJECT_DIR = Path("/content/lichen_growthform_silver")
META_PATH = PROJECT_DIR / "metadata.csv"

metadata = pd.read_csv(META_PATH)

REVIEW_DIR = PROJECT_DIR / "reviewed"
for split in ["good", "reject", "uncertain"]:
    for label in ["crustose", "foliose", "fruticose"]:
        (REVIEW_DIR / split / label).mkdir(parents=True, exist_ok=True)

review_log_path = PROJECT_DIR / "review_log.csv"

if review_log_path.exists():
    review_log = pd.read_csv(review_log_path)
    reviewed_files = set(review_log["file_path"].tolist())
else:
    review_log = pd.DataFrame(columns=["file_path", "growth_form_label", "decision"])
    reviewed_files = set()

to_review = metadata[~metadata["file_path"].isin(reviewed_files)].reset_index(drop=True)

print("Images remaining to review:", len(to_review))

In [ ]:
idx = 0

def save_decision(decision):
    global idx, review_log, to_review

    if idx >= len(to_review):
        clear_output()
        print("Review complete.")
        return

    row = to_review.iloc[idx]
    src = Path(row["file_path"])
    label = row["growth_form_label"]

    dst_dir = REVIEW_DIR / decision / label
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / src.name

    if src.exists() and not dst.exists():
        shutil.copy2(src, dst)

    new_row = pd.DataFrame([{
        "file_path": row["file_path"],
        "growth_form_label": label,
        "decision": decision,
        "scientific_name_trait": row.get("scientific_name_trait", ""),
        "gbif_scientific_name": row.get("gbif_scientific_name", ""),
        "image_url": row.get("image_url", "")
    }])

    review_log = pd.concat([review_log, new_row], ignore_index=True)
    review_log.to_csv(review_log_path, index=False)

    idx += 1
    show_next()

def show_next():
    clear_output(wait=True)

    if idx >= len(to_review):
        print("Review complete.")
        return

    row = to_review.iloc[idx]
    img_path = Path(row["file_path"])

    print(f"Image {idx+1} of {len(to_review)}")
    print("Current label:", row["growth_form_label"])
    print("Trait species:", row.get("scientific_name_trait", ""))
    print("GBIF species:", row.get("gbif_scientific_name", ""))

    try:
        img = Image.open(img_path)
        plt.figure(figsize=(8, 8))
        plt.imshow(img)
        plt.axis("off")
        plt.show()
    except Exception as e:
        print("Could not open image:", e)

    good_btn = widgets.Button(description="Good", button_style="success")
    reject_btn = widgets.Button(description="Reject", button_style="danger")
    uncertain_btn = widgets.Button(description="Uncertain", button_style="warning")
    skip_btn = widgets.Button(description="Skip", button_style="")

    good_btn.on_click(lambda b: save_decision("good"))
    reject_btn.on_click(lambda b: save_decision("reject"))
    uncertain_btn.on_click(lambda b: save_decision("uncertain"))

    def skip(_):
        global idx
        idx += 1
        show_next()

    skip_btn.on_click(skip)

    display(widgets.HBox([good_btn, reject_btn, uncertain_btn, skip_btn]))

show_next()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil
from pathlib import Path

src = Path("/content/lichen_growthform_silver")
dst = Path("/content/drive/MyDrive/EcoSignal/lichen_growthform_silver")

dst.parent.mkdir(parents=True, exist_ok=True)

if dst.exists():
    print("Drive folder already exists:", dst)
else:
    shutil.copytree(src, dst)
    print("Copied to:", dst)

In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/EcoSignal/lichen_growthform_silver")
META_PATH = PROJECT_DIR / "metadata.csv"
REVIEW_DIR = PROJECT_DIR / "reviewed"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/content/drive/MyDrive/EcoSignal/lichen_growthform_silver")
RAW_IMG_DIR = PROJECT_DIR / "images_raw"
META_PATH = PROJECT_DIR / "metadata.csv"
REVIEW_DIR = PROJECT_DIR / "reviewed"

print("Project exists:", PROJECT_DIR.exists())
print("Metadata exists:", META_PATH.exists())
print("Raw images folder exists:", RAW_IMG_DIR.exists())

In [ ]:
for label in ["crustose", "foliose", "fruticose"]:
    folder = RAW_IMG_DIR / label
    count = len(list(folder.glob("*.jpg"))) if folder.exists() else 0
    print(label, count)

total = sum(len(list((RAW_IMG_DIR / label).glob("*.jpg"))) for label in ["crustose", "foliose", "fruticose"])
print("Total images:", total)

Growth Form Model

In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

DATA_DIR = RAW_IMG_DIR

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical"
)

class_names = train_ds.class_names
print("Classes:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(500).prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

In [ ]:
from tensorflow.keras import layers, models

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
])

base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3),
    pooling="avg"
)

base_model.trainable = False

inputs = layers.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)
x = base_model(x, training=False)
x = layers.Dropout(0.35)(x)
outputs = layers.Dense(3, activation="softmax")(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=4,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(PROJECT_DIR / "growthform_effnet_baseline.keras"),
        monitor="val_accuracy",
        save_best_only=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks
)

In [ ]:
base_model.trainable = True

# Freeze most layers, fine-tune only the top part
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    callbacks=callbacks
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(xticks_rotation=45)
plt.show()

Exporting mistakes

In [ ]:
import numpy as np
import pandas as pd
import shutil
from pathlib import Path
from PIL import Image
import tensorflow as tf

PROJECT_DIR = Path("/content/drive/MyDrive/EcoSignal/lichen_growthform_silver")
DATA_DIR = PROJECT_DIR / "images_raw"
REVIEW_ERRORS_DIR = PROJECT_DIR / "model_review_errors"
REVIEW_ERRORS_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

val_ds_paths = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=1,
    label_mode="categorical",
    shuffle=False
)

class_names = val_ds_paths.class_names
file_paths = val_ds_paths.file_paths

rows = []

for i, (image, label) in enumerate(val_ds_paths):
    pred = model.predict(image, verbose=0)[0]
    true_idx = int(np.argmax(label.numpy()[0]))
    pred_idx = int(np.argmax(pred))
    confidence = float(np.max(pred))

    true_label = class_names[true_idx]
    pred_label = class_names[pred_idx]
    src = Path(file_paths[i])

    rows.append({
        "file_path": str(src),
        "true_label": true_label,
        "pred_label": pred_label,
        "confidence": confidence,
        "correct": true_label == pred_label
    })

    if true_label != pred_label:
        dst_dir = REVIEW_ERRORS_DIR / f"true_{true_label}_pred_{pred_label}"
        dst_dir.mkdir(parents=True, exist_ok=True)
        dst = dst_dir / f"conf_{confidence:.2f}_{src.name}"
        shutil.copy2(src, dst)

results_df = pd.DataFrame(rows)
results_df.to_csv(PROJECT_DIR / "growthform_val_predictions.csv", index=False)

print("Saved prediction table to:", PROJECT_DIR / "growthform_val_predictions.csv")
print("Copied misclassified images to:", REVIEW_ERRORS_DIR)
print(results_df["correct"].value_counts())

In [ ]:
LOW_CONF_DIR = PROJECT_DIR / "model_review_low_confidence"
LOW_CONF_DIR.mkdir(parents=True, exist_ok=True)

low_conf = results_df.sort_values("confidence").head(50)

for _, row in low_conf.iterrows():
    src = Path(row["file_path"])
    dst_dir = LOW_CONF_DIR / f"true_{row['true_label']}_pred_{row['pred_label']}"
    dst_dir.mkdir(parents=True, exist_ok=True)

    dst = dst_dir / f"conf_{row['confidence']:.2f}_{src.name}"
    shutil.copy2(src, dst)

print("Copied 50 lowest-confidence images to:", LOW_CONF_DIR)